# Module 18: Model Validation & Registration

**Snowpark ML Fundamentals — Day 1 Afternoon Session 2**

---

## Learning Objectives

Modules 10–11 showed you how to version and promote models mechanically.
Today's focus: **How do you DECIDE whether a new model should replace
the current production model?**

| Concept | What You'll Learn |
|---------|-------------------|
| Holdout validation | Three-way split: train / validation / holdout |
| Champion/Challenger | Formal head-to-head on the same holdout set |
| Programmatic promotion | Threshold-based decision — no human bias |

## Estimated Time: 45 minutes

---

## 18.1 Setup: Simulate a Production Model (5 min)

**Scenario:** A churn prediction model has been in production for 2 months.
You've been asked to train a better one. But first — how do you prove
the new model actually IS better?

We'll create a baseline "production" model and then try to beat it.

In [ ]:
# --- Setup (given — just run this cell) ---
import sys
sys.path.insert(0, '../../../src')

import logging
import warnings

import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

logger = logging.getLogger("module_18")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

from snowflake.snowpark import functions as F

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier
from snowpark_fundamentals.modeling.tuning import (
    randomized_search_cv, get_best_model_params, get_search_results,
)
from snowpark_fundamentals.registry.model_registry import (
    get_registry, log_model, delete_model,
    get_model_by_alias, set_model_alias, set_default_version,
    set_model_comment, get_model_metrics, list_versions,
    compare_model_versions, load_model_and_predict,
)

session = create_session(env_path='../../../.env')
logger.info("Session created.")

In [ ]:
# --- Schema isolation ---
STUDENT_NAME = "YOUR_NAME"  # <-- CHANGE THIS to your name (e.g., "MARIA")

session.sql("USE ROLE MLDS_ROLE_D").collect()
session.sql("USE DATABASE MLDS_D").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {STUDENT_NAME}_ONSITE_TRAINING").collect()
session.sql(f"USE SCHEMA {STUDENT_NAME}_ONSITE_TRAINING").collect()

# Generate dataset
churn_df = create_customer_churn_dataset(session, n_rows=5000)

# --- THREE-WAY SPLIT ---
# train (60%): for hyperparameter search with cross-validation
# validation (20%): for non-holdout sanity checks and baseline reporting
# holdout (20%): for FINAL promotion gating only
train_test_df, holdout_df = split_data(churn_df, train_ratio=0.8)
train_df, validation_df = split_data(train_test_df, train_ratio=0.75)

logger.info(f"Train: {train_df.count()} | Validation: {validation_df.count()} | Holdout: {holdout_df.count()}")
logger.info("Rule: Never use the holdout set for tuning, selection, or iterative debugging.")


In [ ]:
# --- Feature columns ---
FEATURE_COLS = [
    'AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'SUPPORT_TICKETS',
    'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS', 'TOTAL_CHARGES',
]
LABEL_COL = 'CHURNED'

# Registry setup
current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

MODEL_NAME = f"{STUDENT_NAME}_CHURN_VALIDATED"

# Cleanup
try:
    delete_model(registry, MODEL_NAME)
except Exception:
    pass

# --- Create "current production" model (simulates what's already deployed) ---
champion_model = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    model_params={'n_estimators': 50, 'max_depth': 4},  # intentionally weak
)
champion_validation_preds = predict(champion_model, validation_df)
champion_validation_metrics = evaluate_binary_classifier(
    champion_validation_preds, LABEL_COL, 'PREDICTION'
)

log_model(
    registry=registry,
    model=champion_model.to_xgboost(),
    model_name=MODEL_NAME,
    version_name='V1',
    sample_input=validation_df.select(FEATURE_COLS).limit(10),
    metrics=champion_validation_metrics,
)
set_model_alias(registry, MODEL_NAME, 'V1', 'production')
set_default_version(registry, MODEL_NAME, 'V1')
set_model_comment(
    registry, MODEL_NAME,
    "V1: Baseline production model. XGBoost n_estimators=50, max_depth=4. "
    "Stored with pre-holdout validation metrics.",
    version_name='V1',
)

logger.info(f"Production model registered: {MODEL_NAME} / V1")
logger.info(f"Validation metrics (validation_df): {champion_validation_metrics}")
logger.info("Alias 'production' and default version set on V1.")


---

## 18.2 Holdout vs Cross-Validation (10 min)

**Key insight:**
- **Cross-validation** happens inside `train_df` during hyperparameter search
- **Validation split** (`validation_df`) gives one extra non-holdout check after tuning
- **Holdout** is the final promotion gate on truly unseen data

CV scores can still look optimistic because the search process is tuned against them.
The holdout set has NEVER been used. It's the final decision surface.

Best-practice flow for this notebook:
1. Search on `train_df` with CV
2. Sanity-check the chosen params on `validation_df`
3. Retrain on all non-holdout data (`train_test_df`)
4. Compare champion vs challenger on `holdout_df`

Your turn — find better hyperparameters via CV, then validate correctly.


In [ ]:
# TODO 1: Run RandomizedSearchCV to find better hyperparameters
# Use train_df (NOT holdout_df!), cv=3, scoring='f1', n_iter=8
# Hint: See notebook 15 — use randomized_search_cv()
#   randomized_search_cv(train_df, FEATURE_COLS, LABEL_COL,
#       model_type='xgboost', n_iter=8, cv=3, scoring='f1',
#       param_distributions={
#           'n_estimators': [50, 100, 150, 200],
#           'max_depth': [3, 5, 7, 9],
#           'learning_rate': [0.01, 0.05, 0.1, 0.2],
#       })

search_cv = ____

best_params = get_best_model_params(search_cv)
logger.info(f"Best CV F1: {best_params['best_score']:.4f}")
logger.info(f"Best params: {best_params['best_params']}")

In [ ]:
# TODO 2: Train a challenger with the best params using a disciplined validation flow
# Best practice:
#   1. Fit a candidate on train_df using the best CV params
#   2. Evaluate that candidate once on validation_df
#   3. Refit the final challenger on train_test_df (all non-holdout data)
#   4. Evaluate the final challenger on holdout_df
# Hint: Use train_model() twice with model_params=best_params['best_params']

candidate_model = ____
candidate_validation_preds = predict(candidate_model, validation_df)
candidate_validation_metrics = evaluate_binary_classifier(
    candidate_validation_preds, LABEL_COL, 'PREDICTION'
)

challenger_model = ____
challenger_preds = predict(challenger_model, holdout_df)
challenger_metrics = evaluate_binary_classifier(challenger_preds, LABEL_COL, 'PREDICTION')

logger.info(f"Candidate validation F1: {candidate_validation_metrics['f1_score']:.4f}")
logger.info(f"Final challenger holdout F1: {challenger_metrics['f1_score']:.4f}")
logger.info(f"(CV F1 was {best_params['best_score']:.4f}; holdout is the promotion gate)")


In [ ]:
# --- Validation ---
assert search_cv is not None, "TODO 1: search_cv must be run"
assert isinstance(candidate_validation_metrics, dict), "TODO 2: candidate_validation_metrics should be a dict"
assert 'f1_score' in candidate_validation_metrics, "TODO 2: candidate_validation_metrics should have f1_score"
assert isinstance(challenger_metrics, dict), "TODO 2: challenger_metrics should be a dict"
assert 'f1_score' in challenger_metrics, "TODO 2: challenger_metrics should have f1_score"
logger.info("Section 18.2: All validations passed!")


---

## 18.3 Champion/Challenger Comparison (15 min)

**The pattern:**
1. Load the current **champion** from the registry by its `'production'` alias
2. Score the **holdout** set with the champion
3. Score the **holdout** set with the challenger
4. Compare metrics — promote ONLY if the challenger beats the champion by a meaningful margin

The threshold prevents "flip-flopping" between models that are essentially equivalent.
A 1% F1 improvement is a reasonable starting point.

In [ ]:
# --- Load champion from registry and score holdout ---
champion_holdout_preds = load_model_and_predict(
    registry,
    MODEL_NAME,
    'production',
    holdout_df.select(FEATURE_COLS + [LABEL_COL]),
)
champion_holdout_metrics = evaluate_binary_classifier(
    champion_holdout_preds, LABEL_COL, 'PREDICTION',
)

logger.info("Champion (V1) holdout metrics:")
for k, v in champion_holdout_metrics.items():
    logger.info(f"  {k}: {v:.4f}")

logger.info(f"\nChallenger holdout metrics:")
for k, v in challenger_metrics.items():
    logger.info(f"  {k}: {v:.4f}")


In [ ]:
# TODO 3: Compare champion vs challenger and decide whether to promote
# Rules:
#   - If challenger F1 > champion F1 + PROMOTION_THRESHOLD: PROMOTE
#   - Otherwise: KEEP current champion
# If promoting:
#   1. log_model() — register challenger as 'V2' with metrics=challenger_metrics
#   2. set_model_alias() — move 'production' alias to V2
#   3. set_default_version() — keep the default version aligned with production
#   4. set_model_comment() — explain the promotion rationale (include both F1 scores)
# Hint: Use challenger_model.to_xgboost() for log_model

PROMOTION_THRESHOLD = 0.01
improvement = challenger_metrics['f1_score'] - champion_holdout_metrics['f1_score']

logger.info(f"F1 improvement: {improvement:.4f} (threshold: {PROMOTION_THRESHOLD})")

if improvement > PROMOTION_THRESHOLD:
    logger.info("DECISION: Promote challenger to production\n")

    # Step 1: Register challenger
    ____

    # Step 2: Move production alias
    ____

    # Step 3: Keep default version aligned
    ____

    # Step 4: Document the promotion
    ____

    promoted = True
else:
    logger.info(f"DECISION: Keep champion (improvement {improvement:.4f} < threshold {PROMOTION_THRESHOLD})")
    promoted = False

# Show final state
versions_df = list_versions(registry, MODEL_NAME)
logger.info("\nRegistry state:")
logger.info(versions_df[['name', 'aliases']].to_string())


In [ ]:
# --- Validation ---
assert isinstance(promoted, bool), "TODO 3: promoted should be True or False"
versions_df = list_versions(registry, MODEL_NAME)
assert len(versions_df) >= 1, "At least one version should exist"
logger.info("Section 18.3: All validations passed!")

---

## 18.4 "Laptop to Production" — What Changes? (10 min)

Everything you just did manually maps directly to production automation:

| Notebook Step | Production Equivalent |
|--------------|----------------------|
| `randomized_search_cv()` | Scheduled retraining job |
| `predict(challenger, holdout)` | Validation step in stored procedure |
| `if improvement > threshold` | Programmatic gate — no human in the loop |
| `set_model_alias('production')` | Zero-downtime promotion |
| `set_model_comment()` | Audit trail for compliance |

Here's what the stored procedure looks like — read through it.
This is NOT a TODO — just observe how your notebook code translates.

In [ ]:
# --- Production stored procedure (read-only — observe the pattern) ---
champion_challenger_sp = f"""
CREATE OR REPLACE PROCEDURE {STUDENT_NAME}_CHAMPION_CHALLENGER(
    MODEL_NAME VARCHAR,
    TRAINING_TABLE VARCHAR,
    THRESHOLD FLOAT DEFAULT 0.01
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'snowflake-ml-python')
HANDLER = 'run'
AS $$
def run(session, model_name: str, training_table: str, threshold: float) -> str:
    from snowflake.ml.registry import Registry
    from snowflake.ml.modeling.xgboost import XGBClassifier

    # 1. Load fresh data and split
    df = session.table(training_table)
    train_df, holdout_df = df.random_split([0.8, 0.2], seed=42)

    feature_cols = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES',
                    'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK',
                    'NUM_DEPENDENTS', 'TOTAL_CHARGES']

    # 2. Train challenger
    challenger = XGBClassifier(
        input_cols=feature_cols, label_cols=['CHURNED'],
        output_cols=['PREDICTION'],
        n_estimators=150, max_depth=7,
    )
    challenger.fit(train_df)

    # 3. Load champion by alias and normalize registry prediction output
    registry = Registry(session=session)
    champion = registry.get_model(model_name).version('production')
    champion_preds = champion.run(
        holdout_df.select(feature_cols + ['CHURNED']),
        function_name='predict',
    )
    if 'PREDICTION' not in champion_preds.columns and 'output_feature_0' in champion_preds.columns:
        champion_preds = champion_preds.with_column_renamed('output_feature_0', 'PREDICTION')
    elif 'PREDICTION' not in champion_preds.columns and 'OUTPUT_FEATURE_0' in champion_preds.columns:
        champion_preds = champion_preds.with_column_renamed('OUTPUT_FEATURE_0', 'PREDICTION')
    challenger_preds = challenger.predict(holdout_df)

    # 4. Compare (simplified — production code would use proper metrics)
    # ... compute F1 scores, compare, promote if better ...

    return f'Champion/challenger check completed for {{model_name}}'
$$;
"""

logger.info("Stored procedure structure:")
logger.info("  1. Load fresh data → 2. Train challenger → 3. Load champion by alias")
logger.info("  4. Normalize registry output → 5. Compare on holdout → 6. Promote if better")
logger.info("\nEvery step uses functions you already know.")
logger.info("The only new thing is the SQL wrapper.")


---

## Key Takeaways

1. **Three-way split:** Train → Validation → Holdout. Cross-validation happens inside the training split during tuning.

2. **Champion/challenger** is the standard production promotion pattern.
   The current model is the "champion." Any new candidate is a "challenger."

3. **Programmatic thresholds** remove human bias. A 1% F1 improvement
   threshold prevents flip-flopping between similar models.

4. **The Model Registry is the single source of truth.** The `'production'`
   alias should always point to the best validated model. Keep the
   default version aligned, but load application code by alias.

**Next session:** The model is deployed. How do we know it's still working?
→ Module 19: Deployment, Serving & Monitoring


In [ ]:
# --- Cleanup ---
delete_model(registry, MODEL_NAME)
logger.info(f"Cleaned up {MODEL_NAME}")

In [ ]:
session.close()